# psfFlux / scienceFlux / templateFlux — colour-coded by detector (CCD)

## Motivation

In the DIA pipeline:
```
psfFlux  =  scienceFlux  −  templateFlux
```
where `templateFlux_est = scienceFlux − psfFlux` is the implicit template
contribution reconstructed per epoch.

This notebook displays the three raw fluxes **in nJy** (no normalisation)
for the top-N DIA objects (**src only**, no forced photometry), with each
measurement **colour-coded by detector number (CCD)** using a discrete
`tab20 + tab20b` palette (same convention as `04c02_relativeFlux_withfp_showdetectors.ipynb`).

### Layout per DIA object
```
  u  |  g  |  r  |  i  |  z  |  y  |  all-bands
  ─────────────────────────────────────────────────
                  CCD legend strip
```

Each flux panel shows:
- **● scienceFlux**       (filled circle)
- **□ psfFlux**           (open square, edge = CCD colour)
- **◇ templateFlux_est**  (open diamond, edge = CCD colour)

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab / IN2P3 / CNRS
- **Created from:** `04c02_relativeFlux_withfp_showdetectors.ipynb` + `07_psfFlux_scienceFlux.ipynb`
- **Last update:** 2026-04-12
- **Subject:** Fink alert broker — Rubin LSST DIA photometry diagnostics vs CCD detector

## 1. Imports & configuration

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from astropy.time import Time

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")

In [ ]:
# ── Notebook tag & paths ──────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_07c"
DIR_DATA_IN = "data_FINK_BLOCK_LC_03"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)

FILE_BUTLER = os.path.join(DIR_DATA_IN, "src_joined_butler.parquet")
FILE_CONSDB = os.path.join(DIR_DATA_IN, "src_joined_consdb.parquet")

N_TOP = 20

BANDS = list("ugrizy")
BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}

# ── Column names ──────────────────────────────────────────────────────────────
COL_OBJ = "r:diaObjectId"
COL_SRC = "r:diaSourceId"
COL_MJD = "r:midpointMjdTai"
COL_BAND = "r:band"
COL_PSF = "r:psfFlux"
COL_PSFERR = "r:psfFluxErr"
COL_DET = "r:detector"  # CCD number — must be present in Butler-joined parquet

SCIENCE_FLUX_CANDIDATES = [
    "r:scienceFlux",
    "r:psfScience",
    "r:psfScienceFlux",
    "scienceFlux",
    "psfScience",
    "psfScienceFlux",
]
SCIENCE_ERR_CANDIDATES = [
    "r:scienceSigma",
    "r:scienceFluxErr",
    "r:psfScienceSigma",
    "r:psfScienceFluxErr",
    "scienceSigma",
    "scienceFluxErr",
    "psfScienceSigma",
]

TEMPLATE_FLUX_CANDIDATES = [
    "psfTemplateFlux",
    "r:templateFlux",
    "r:template",
    "r:psfScienceFlux",
    "templateFlux",
    "template",
    "psfTemplateFlux",
]
TEMPLATE_ERR_CANDIDATES = [
    "r:templaeSigma",
    "r:templateFluxErr",
    "r:psfTemplateSigma",
    "r:psfTemplateFluxErr",
    "templateSigma",
    "templateFluxErr",
    "psfTemplateSigma",
]

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print(f"Input directory  : {os.path.abspath(DIR_DATA_IN)}")
print(f"Figure directory : {os.path.abspath(DIR_FIGS)}")
print(f"N_TOP            : {N_TOP}")

## 2. Load data

In [ ]:
if os.path.exists(FILE_BUTLER):
    df = pd.read_parquet(FILE_BUTLER)
    src_label = "butler"
    print(f"Loaded butler file : {FILE_BUTLER}")
elif os.path.exists(FILE_CONSDB):
    df = pd.read_parquet(FILE_CONSDB)
    src_label = "consdb"
    print(f"Butler file not found. Loaded consDb : {FILE_CONSDB}")
else:
    raise FileNotFoundError(
        f"Neither {FILE_BUTLER} nor {FILE_CONSDB} found.\n"
        "Run notebook 03_associateFinkAlerts-RubinVisits.ipynb first."
    )
print(f"Shape : {df.shape[0]:,} rows x {df.shape[1]} columns  |  source : {src_label}")

## 3. Schema inspection — detect scienceFlux & detector columns

In [ ]:
print("All columns:")
for col in sorted(df.columns):
    print(f"  {col:50s}  dtype={df[col].dtype}")

In [ ]:
# ── scienceFlux ───────────────────────────────────────────────────────────────
COL_SCI = None
COL_SCIERR = None
for c in SCIENCE_FLUX_CANDIDATES:
    if c in df.columns:
        COL_SCI = c
        break
for c in SCIENCE_ERR_CANDIDATES:
    if c in df.columns:
        COL_SCIERR = c
        break

has_science = COL_SCI is not None
has_sci_err = COL_SCIERR is not None
print(f"scienceFlux    : {COL_SCI}")
print(f"scienceFluxErr : {COL_SCIERR}")

# ── Detector column ───────────────────────────────────────────────────────────
if COL_DET not in df.columns:
    raise KeyError(
        f"Detector column '{COL_DET}' not found.\n"
        "This notebook requires Butler-joined data (src_joined_butler.parquet)."
    )
print(f"Detector column: '{COL_DET}'  — {df[COL_DET].nunique()} unique CCDs")

In [ ]:
COL_TEMP = None
COL_TEMPERR = None

for candidate in TEMPLATE_FLUX_CANDIDATES:
    if candidate in df.columns:
        COL_TEMP = candidate
        print(f"Found template flux column  : '{COL_TEMP}'")
        break

for candidate in TEMPLATE_ERR_CANDIDATES:
    if candidate in df.columns:
        COL_TEMPERR = candidate
        print(f"Found template flux err col : '{COL_TEMPERR}'")
        break

if COL_TEMP is None:
    print("WARNING: No templateFlux-like column found.")
    temp_cols = [c for c in df.columns if "temp" in c.lower()]
    print(f"Columns with 'temp': {temp_cols}")
else:
    n_valid = df[COL_TEMP].notna().sum()
    print(f"Non-NaN values: {n_valid:,} / {len(df):,}  ({100 * n_valid / len(df):.1f}%)")

has_template = COL_TEMP is not None
has_template_err = COL_TEMPERR is not None

print(f"\n  psfFlux     : {COL_PSF}")
print(f"  scienceFlux : {COL_SCI}")
print(f"  scienceErr  : {COL_SCIERR}")
print(f"  templateFlux: {COL_TEMP}")
print(f"  templateErr : {COL_TEMPERR}")

## 4. Build detector colour palette (tab20 + tab20b, same as 04c02)

In [ ]:
# Sort detectors by decreasing alert count: most active CCDs → most distinct colours
det_counts = df[COL_DET].value_counts()  # descending
all_det_ids = det_counts.index.tolist()
n_det = len(all_det_ids)
print(f"Total unique detector IDs : {n_det}")
print("Top-10 most active CCDs:")
print(det_counts.head(10).to_string())

tab20_colors = [mcm.tab20(i) for i in np.linspace(0, 1, 20, endpoint=False)]
tab20b_colors = [mcm.tab20b(i) for i in np.linspace(0, 1, 20, endpoint=False)]
palette_base = tab20_colors + tab20b_colors  # 40 colours

DET_PALETTE = {det_id: palette_base[i % len(palette_base)] for i, det_id in enumerate(all_det_ids)}

print(f"\nPalette built: {len(palette_base)} base colours for {n_det} detectors.")
if n_det > len(palette_base):
    print(f"  WARNING: {n_det - len(palette_base)} detectors will share colours (cycled).")

In [ ]:
# Visual palette check (top-40 CCDs)
n_show = min(40, n_det)
fig_pal, ax_pal = plt.subplots(figsize=(14, 1), constrained_layout=True)
for i, det_id in enumerate(all_det_ids[:n_show]):
    ax_pal.add_patch(plt.Rectangle((i, 0), 1, 1, color=DET_PALETTE[det_id]))
    ax_pal.text(
        i + 0.5, 0.5, str(det_id), ha="center", va="center", fontsize=6.5, color="k", fontweight="bold"
    )
ax_pal.set_xlim(0, n_show)
ax_pal.set_ylim(0, 1)
ax_pal.set_xticks([])
ax_pal.set_yticks([])
ax_pal.set_title(
    f"Detector colour palette — top-{n_show} CCDs sorted by alert count (left = most active)", fontsize=10
)
savefig("detector_colour_palette")
plt.show()

## 5. Rank DIA objects by alert count

In [ ]:
alert_counts = df.groupby(COL_OBJ).size().rename("n_alerts").sort_values(ascending=False)
print(f"Total unique DIA objects : {len(alert_counts):,}")
print(f"Top {N_TOP}:")
print(alert_counts.head(N_TOP).to_string())

In [ ]:
top_objects = alert_counts.head(N_TOP).index.tolist()
df_top = df[df[COL_OBJ].isin(top_objects)].copy()
df_top[COL_MJD] = df_top[COL_MJD].astype(float)

if has_science:
    df_top["_tmpl"] = df_top[COL_SCI].astype(float) - df_top[COL_PSF].astype(float)

print(f"Rows kept : {len(df_top):,}")

## 6. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def mjd_to_datestr(mjd_array):
    t = Time(np.asarray(mjd_array, dtype=float), format="mjd", scale="tai")
    return [tt.strftime("%Y-%m-%d") for tt in t]


def _add_date_axis(ax, dt_array, t0_mjd):
    """Secondary top x-axis with calendar dates."""
    dt_finite = dt_array[np.isfinite(dt_array)]
    if len(dt_finite) == 0:
        return
    dt_min, dt_max = float(dt_finite.min()), float(dt_finite.max())
    if dt_max <= dt_min:
        tick_dt = np.array([dt_min])
    else:
        n_ticks = min(5, max(2, int((dt_max - dt_min) / 10) + 2))
        tick_dt = np.linspace(dt_min, dt_max, n_ticks)
    tick_dates = mjd_to_datestr(t0_mjd + tick_dt)
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(tick_dt)
    ax2.set_xticklabels(tick_dates, rotation=35, ha="left", fontsize=7)
    ax2.tick_params(axis="x", length=3)


def det_color(det_id):
    """Return the colour for a given detector ID."""
    return DET_PALETTE.get(det_id, "#888888")


def build_det_legend_handles(det_ids_present):
    """Build legend handles for the set of CCD IDs actually visited by this light curve."""
    handles = []
    for det_id in sorted(det_ids_present):
        c = det_color(det_id)
        h = Line2D(
            [0],
            [0],
            marker="o",
            color="none",
            markerfacecolor=c,
            markeredgecolor=c,
            markersize=6,
            label=f"CCD {det_id}",
        )
        handles.append(h)
    return handles


def _shared_lim(arrays, margin=0.05):
    vals = np.concatenate([np.asarray(a, dtype=float).ravel() for a in arrays])
    finite = vals[np.isfinite(vals)]
    if len(finite) == 0:
        return None
    vmin, vmax = finite.min(), finite.max()
    span = max(vmax - vmin, 1e-6)
    return vmin - margin * span, vmax + margin * span


print("Utility functions defined.")

## 6b. Per-object pixel/flux table — `get_diaobject_table()`

This function extracts a **flat, analysis-ready DataFrame** for a single `diaObjectId`
from the already-loaded DataFrame `df` (the parquet file from Section 2).

It uses the column names resolved in Section 3 (`COL_SCI`, `COL_TEMP`, etc.) and
returns a clean table with standardised short column names.

| column | source column | description |
|---|---|---|
| `mjd` | `r:midpointMjdTai` | Midpoint of exposure (TAI, MJD) |
| `band` | `r:band` | Photometric band (`u/g/r/i/z/y`) |
| `visit` | `r:visit` | Butler visitId (Int64, nullable) |
| `detector` | `r:detector` | CCD detector number (Int64, nullable) |
| `x` | `r:x` | Centroid x pixel coordinate on CCD |
| `y` | `r:y` | Centroid y pixel coordinate on CCD |
| `psfFlux` | `r:psfFlux` | PSF flux of the difference-image source (nJy) |
| `psfFluxErr` | `r:psfFluxErr` | Uncertainty on psfFlux (nJy) |
| `scienceFlux` | `COL_SCI` | Flux on the science image (nJy); NaN if unavailable |
| `scienceFluxErr` | `COL_SCIERR` | Uncertainty on scienceFlux (nJy); NaN if unavailable |
| `templateFlux` | `COL_TEMP` | Flux on the template image (nJy); NaN if unavailable |
| `templateFluxErr` | `COL_TEMPERR` | Uncertainty on templateFlux (nJy); NaN if unavailable |

In [ ]:
def get_diaobject_table(diaObjectId, source_df=None) -> pd.DataFrame:
    """
    Return a flat analysis-ready DataFrame for a single diaObjectId.

    Parameters
    ----------
    diaObjectId : int or str
    source_df   : pd.DataFrame or None — defaults to module-level `df`.

    Returns
    -------
    pd.DataFrame with columns:
        diaObj, diaSrc, mjd, band, visit (Int64), detector (Int64), x, y,
        psfFlux, psfFluxErr, scienceFlux, scienceFluxErr,
        templateFlux, templateFluxErr, day_obs, month_obs
    """
    _src = source_df if source_df is not None else df
    mask = _src[COL_OBJ].astype(str) == str(diaObjectId)
    df_obj = _src[mask].copy()

    empty_cols = [
        "diaObj",
        "diaSrc",
        "mjd",
        "band",
        "visit",
        "detector",
        "x",
        "y",
        "psfFlux",
        "psfFluxErr",
        "scienceFlux",
        "scienceFluxErr",
        "templateFlux",
        "templateFluxErr",
        "day_obs",
        "month_obs",
    ]
    if df_obj.empty:
        return pd.DataFrame(columns=empty_cols)

    out = pd.DataFrame()
    out["diaObj"] = df_obj[COL_OBJ].values
    out["diaSrc"] = df_obj[COL_SRC].values
    out["mjd"] = pd.to_numeric(df_obj[COL_MJD], errors="coerce")
    out["band"] = df_obj[COL_BAND].values
    out["psfFlux"] = pd.to_numeric(df_obj[COL_PSF], errors="coerce")
    out["psfFluxErr"] = pd.to_numeric(df_obj[COL_PSFERR], errors="coerce")

    for col_in, col_out in (("r:visit", "visit"), ("r:detector", "detector")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce").astype("Int64")
        else:
            out[col_out] = pd.array([pd.NA] * len(df_obj), dtype="Int64")

    for col_in, col_out in (("r:x", "x"), ("r:y", "y")):
        if col_in in df_obj.columns:
            out[col_out] = pd.to_numeric(df_obj[col_in], errors="coerce")
        else:
            out[col_out] = np.nan

    out["scienceFlux"] = (
        pd.to_numeric(df_obj[COL_SCI], errors="coerce")
        if COL_SCI is not None and COL_SCI in df_obj.columns
        else np.nan
    )
    out["scienceFluxErr"] = (
        pd.to_numeric(df_obj[COL_SCIERR], errors="coerce")
        if COL_SCIERR is not None and COL_SCIERR in df_obj.columns
        else np.nan
    )
    out["templateFlux"] = (
        pd.to_numeric(df_obj[COL_TEMP], errors="coerce")
        if COL_TEMP is not None and COL_TEMP in df_obj.columns
        else np.nan
    )
    out["templateFluxErr"] = (
        pd.to_numeric(df_obj[COL_TEMPERR], errors="coerce")
        if COL_TEMPERR is not None and COL_TEMPERR in df_obj.columns
        else np.nan
    )

    out = (
        out[
            [
                "diaObj",
                "diaSrc",
                "mjd",
                "band",
                "visit",
                "detector",
                "x",
                "y",
                "psfFlux",
                "psfFluxErr",
                "scienceFlux",
                "scienceFluxErr",
                "templateFlux",
                "templateFluxErr",
            ]
        ]
        .sort_values(["mjd", "band"], na_position="last")
        .reset_index(drop=True)
    )
    out["day_obs"] = out["visit"] // 100000
    out["month_obs"] = out["visit"] // 10000000
    return out


print("get_diaobject_table() defined.")

### 6b-demo — Quick test of `get_diaobject_table()`

In [ ]:
_test_oid = top_objects[1]
df_test = get_diaobject_table(_test_oid)
print(f"diaObjectId {_test_oid}  →  {len(df_test)} rows")
print(f"  bands present   : {sorted(df_test['band'].dropna().unique())}")
print(f"  visit range     : {df_test['visit'].dropna().min()} … {df_test['visit'].dropna().max()}")
print(f"  mjd  range      : {df_test['mjd'].min():.2f} … {df_test['mjd'].max():.2f}")
print(f"  scienceFlux NaN : {df_test['scienceFlux'].isna().sum()} / {len(df_test)}")
print(f"  templateFlux NaN: {df_test['templateFlux'].isna().sum()} / {len(df_test)}")
display(df_test)

## 6c. Load diaObject catalog from file

In [ ]:
DIR_CAT = "data_FINK_BLOCK_LC_01"
_cat_priority = [
    os.path.join(DIR_CAT, "diaobj_catalogue_gaia_stable.csv"),
    os.path.join(DIR_CAT, "diaobj_catalogue.csv"),
]
df_cat_ref = None
for _cat_path in _cat_priority:
    if os.path.exists(_cat_path):
        df_cat_ref = pd.read_csv(_cat_path, low_memory=False)
        df_cat_ref["diaObjectId"] = df_cat_ref["diaObjectId"].astype(str)
        print(f"Loaded catalogue : {_cat_path}  ({len(df_cat_ref)} objects)")
        break
if df_cat_ref is None:
    print("WARNING: diaObject catalogue not found — reference flux lines will be skipped.")

## 7. Main plot function: 3 fluxes vs Δt, colour = CCD

In [ ]:
def plot_3fluxes_by_detector(obj_id, df_obj, axes_flux, ax_legend):
    """
    Plot scienceFlux (●), psfFlux (□) and templateFlux (◇) in nJy
    vs Δt [days], colour-coded by CCD detector number.

    Parameters
    ----------
    obj_id     : int/str    — diaObjectId
    df_obj     : DataFrame  — rows for this object, sorted by MJD
    axes_flux  : list of 7  — band panels (u g r i z y + combined)
    ax_legend  : Axes       — dedicated legend panel (below flux panels)

    Returns
    -------
    t0_date    : str         — ISO date of the first alert
    all_dets   : set         — all CCD IDs seen for this object
    """

    # special with object catalog for flux straight-lines
    oid_str = str(obj_id)
    ref_psf = {}
    ref_sci = {}
    if df_cat_ref is not None:
        mask_cat = df_cat_ref["diaObjectId"].astype(str) == oid_str
        if mask_cat.any():
            cat_row = df_cat_ref[mask_cat].iloc[0]
            for b in BANDS:
                for dest, col in [(ref_psf, f"r:{b}_psfFluxMean"), (ref_sci, f"r:{b}_scienceFluxMean")]:
                    try:
                        fval = float(cat_row[col])
                        dest[b] = fval if np.isfinite(fval) else None
                    except (KeyError, TypeError, ValueError):
                        dest[b] = None

    t0_mjd = float(df_obj[COL_MJD].min())
    t0_date = mjd_to_datestr([t0_mjd])[0]

    band_data = {}
    all_flux = []
    all_dt = []
    all_dets = set()

    for band in BANDS:
        mask = df_obj[COL_BAND] == band
        df_b = df_obj[mask].sort_values(COL_MJD)
        if len(df_b) == 0:
            band_data[band] = None
            continue
        dt = df_b[COL_MJD].values.astype(float) - t0_mjd
        det_ids = df_b[COL_DET].values
        psf = df_b[COL_PSF].values.astype(float)
        psferr = df_b[COL_PSFERR].values.astype(float)
        sci = df_b[COL_SCI].values.astype(float) if has_science else None
        scierr = df_b[COL_SCIERR].values.astype(float) if has_sci_err else None
        temp = df_b[COL_TEMP].values.astype(float) if has_template else None
        temperr = df_b[COL_TEMPERR].values.astype(float) if has_template_err else None

        band_data[band] = dict(
            dt=dt, det_ids=det_ids, psf=psf, psferr=psferr, sci=sci, scierr=scierr, temp=temp, temperr=temperr
        )
        all_dt.append(dt)
        all_flux.append(psf)
        if sci is not None:
            all_flux.append(sci)
        if temp is not None:
            all_flux.append(temp)
        all_dets.update(det_ids.tolist())

    ylim = _shared_lim(all_flux) if all_flux else None
    xlim = _shared_lim(all_dt) if all_dt else None

    def _plot_series_by_det(ax, dt, flux, flux_err, det_ids, fmt, ms, alpha, open_marker=False):
        """Plot each point individually with its CCD colour."""
        for i in range(len(dt)):
            c = det_color(det_ids[i])
            if open_marker:
                ax.errorbar(
                    dt[i],
                    flux[i],
                    yerr=flux_err[i] if flux_err is not None else 0,
                    fmt=fmt,
                    ms=ms,
                    color="none",
                    mec=c,
                    mew=1.1,
                    ecolor=c,
                    elinewidth=0.7,
                    capsize=2,
                    alpha=alpha,
                )
            else:
                ax.errorbar(
                    dt[i],
                    flux[i],
                    yerr=flux_err[i] if flux_err is not None else 0,
                    fmt=fmt,
                    ms=ms,
                    color=c,
                    ecolor=c,
                    elinewidth=0.8,
                    capsize=2,
                    alpha=alpha,
                )

    for idx, band in enumerate(BANDS):
        ax = axes_flux[idx]
        bcolor = BAND_COLORS.get(band, "k")

        if band_data[band] is None:
            ax.text(
                0.5,
                0.5,
                "no data",
                ha="center",
                va="center",
                transform=ax.transAxes,
                color="gray",
                fontsize=8,
            )
            ax.set_title(f"band {band}", fontsize=9, color=bcolor)
            if xlim:
                ax.set_xlim(xlim)
            if ylim:
                ax.set_ylim(ylim)
            ax.set_xlabel("delta-t [days]", fontsize=8)
            continue

        d = band_data[band]
        if d["sci"] is not None:
            _plot_series_by_det(
                ax,
                d["dt"],
                d["sci"],
                d["scierr"] if d["scierr"] is not None else np.zeros_like(d["sci"]),
                d["det_ids"],
                "o",
                ms=5,
                alpha=0.88,
                open_marker=False,
            )
        _plot_series_by_det(
            ax, d["dt"], d["psf"], d["psferr"], d["det_ids"], "s", ms=4, alpha=0.55, open_marker=True
        )
        if d["temp"] is not None:
            _plot_series_by_det(
                ax,
                d["dt"],
                d["temp"],
                np.zeros_like(d["temp"]),
                d["det_ids"],
                "D",
                ms=4,
                alpha=0.55,
                open_marker=True,
            )

        # add object flux per band
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            mag, merr = flux_to_mag(v_sci)
            ax.axhline(v_sci, color=bcolor, lw=1.2, ls="-.", alpha=0.85, label=f"obj: m={mag:.1f} mag")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            mag, merr = flux_to_mag(v_psf)
            ax.axhline(v_psf, color=bcolor, lw=1.2, ls=":", alpha=0.85, label=f"diaobj: m={mag:.1f} mag")

        ax.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
        ax.set_title(f"band {band}  |  sci● psf□ tmpl◇\ncolour = CCD", fontsize=8, color=bcolor)
        ax.set_ylabel("flux [nJy]", fontsize=8)
        ax.set_xlabel("delta-t [days]", fontsize=8)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        _add_date_axis(ax, d["dt"], t0_mjd)

    ax_all = axes_flux[-1]
    for band in BANDS:
        if band_data[band] is None:
            continue
        d = band_data[band]
        if d["sci"] is not None:
            _plot_series_by_det(
                ax_all,
                d["dt"],
                d["sci"],
                d["scierr"] if d["scierr"] is not None else np.zeros_like(d["sci"]),
                d["det_ids"],
                "o",
                ms=3,
                alpha=0.80,
                open_marker=False,
            )
        _plot_series_by_det(
            ax_all, d["dt"], d["psf"], d["psferr"], d["det_ids"], "s", ms=3, alpha=0.45, open_marker=True
        )
        if d["temp"] is not None:
            _plot_series_by_det(
                ax_all,
                d["dt"],
                d["temp"],
                np.zeros_like(d["temp"]),
                d["det_ids"],
                "D",
                ms=3,
                alpha=0.45,
                open_marker=True,
            )

        # horizontal lines with object flux
        bcolor = BAND_COLORS.get(band, "k")
        v_sci = ref_sci.get(band)
        if v_sci is not None:
            ax_all.axhline(v_sci, color=bcolor, lw=0.7, ls="-.", alpha=0.45, label="_nolegend_")
        v_psf = ref_psf.get(band)
        if v_psf is not None:
            ax_all.axhline(v_psf, color=bcolor, lw=0.7, ls=":", alpha=0.45, label="_nolegend_")

    ax_all.axhline(0.0, color="k", lw=0.6, ls=":", alpha=0.4)
    ax_all.set_title("all bands\nsci ● psf □ tmpl ◇  (colour = CCD)", fontsize=9)
    ax_all.set_ylabel("flux [nJy]", fontsize=8)
    ax_all.set_xlabel("delta-t [days]", fontsize=8)
    if xlim:
        ax_all.set_xlim(xlim)
    _add_date_axis(ax_all, df_obj[COL_MJD].values - t0_mjd, t0_mjd)

    # ── Dedicated CCD legend panel ────────────────────────────────────────────
    ax_legend.set_axis_off()
    det_handles = build_det_legend_handles(all_dets)
    type_handles = [
        Line2D([0], [0], marker="o", color="k", markersize=6, label="● scienceFlux (filled)"),
        Line2D(
            [0],
            [0],
            marker="s",
            color="none",
            markeredgecolor="k",
            markersize=6,
            linewidth=0,
            label="□ psfFlux (open)",
        ),
        Line2D(
            [0],
            [0],
            marker="D",
            color="none",
            markeredgecolor="k",
            markersize=6,
            linewidth=0,
            label="◇ templateFlux (open)",
        ),
    ]
    all_handles = type_handles + [mpatches.Patch(color="none", label="─── CCD ───")] + det_handles
    n_cols = max(4, min(12, len(all_dets) // 3 + 4))
    ax_legend.legend(
        handles=all_handles,
        loc="center",
        ncol=n_cols,
        fontsize=7,
        title=f"Marker types & CCD detectors visited  (N_CCD={len(all_dets)})",
        title_fontsize=8,
        frameon=True,
        framealpha=0.8,
    )

    return t0_date, all_dets


print("plot_3fluxes_by_detector() defined.")

## 8. Main loop — one figure per DIA object

Each figure has **2 rows**:
- Row 0 (height ratio 4): 7 flux panels
- Row 1 (height ratio 1): dedicated CCD legend strip

In [ ]:
if not has_science:
    print("WARNING: scienceFlux column not found — only psfFlux will be plotted.")

n_flux_panels = len(BANDS) + 1  # 7 panels

for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id].sort_values(COL_MJD)
    n_alerts = len(df_obj)
    field = df_obj["field"].iloc[0] if "field" in df_obj.columns else ""
    det_list = sorted(df_obj[COL_DET].dropna().unique().tolist())

    fig = plt.figure(figsize=(3.2 * n_flux_panels, 5.5))
    gs = fig.add_gridspec(2, n_flux_panels, height_ratios=[4, 1], hspace=0.35, wspace=0.45)

    axes_flux = [fig.add_subplot(gs[0, k]) for k in range(n_flux_panels)]
    ax_legend = fig.add_subplot(gs[1, :])

    t0_date, all_dets = plot_3fluxes_by_detector(obj_id, df_obj, axes_flux, ax_legend)

    fig.suptitle(
        f"rank #{rank + 1}  |  diaObjectId={obj_id}  |  field={field}  |"
        f"  N_src={n_alerts}  N_CCD={len(all_dets)}  |  t₀={t0_date}\n"
        "scienceFlux ● (filled)   psfFlux □ (open)   templateFlux ◇ (open)"
        "   [nJy]   —   colour = detector (CCD number)",
        fontsize=10,
        y=1.01,
    )

    savefig(f"3flux_bydet_{src_label}_rank{rank + 1:02d}_obj{obj_id}")
    plt.show()

## 9. Summary: median flux levels + CCD census per object and band

In [ ]:
records = []
for rank, obj_id in enumerate(top_objects):
    df_obj = df_top[df_top[COL_OBJ] == obj_id]
    t0_mjd = df_obj[COL_MJD].min()
    t0_date = mjd_to_datestr([t0_mjd])[0]
    all_dets = sorted(df_obj[COL_DET].dropna().unique().tolist())

    row = {
        "rank": rank + 1,
        "diaObjectId": obj_id,
        "n_alerts": len(df_obj),
        "t0_date": t0_date,
        "n_det_total": len(all_dets),
        "det_ids": ",".join(str(d) for d in all_dets),
    }
    for band in BANDS:
        df_b = df_obj[df_obj[COL_BAND] == band]
        if len(df_b) == 0:
            for k in ("n_det", "med_psf", "med_sci", "med_tmpl"):
                row[f"{k}_{band}"] = np.nan
            continue
        row[f"n_det_{band}"] = int(df_b[COL_DET].nunique())
        row[f"med_psf_{band}"] = round(float(np.nanmedian(df_b[COL_PSF].astype(float))), 2)
        if has_science:
            row[f"med_sci_{band}"] = round(float(np.nanmedian(df_b[COL_SCI].astype(float))), 2)
            row[f"med_tmpl_{band}"] = round(float(np.nanmedian(df_b["_tmpl"].astype(float))), 2)
        else:
            row[f"med_sci_{band}"] = np.nan
            row[f"med_tmpl_{band}"] = np.nan
    records.append(row)

df_summary = pd.DataFrame(records)
print("Median flux levels [nJy] and CCD census per object and band:")
display(df_summary)

In [ ]:
out = os.path.join(DIR_FIGS, f"3flux_bydet_summary_{src_label}.csv")
df_summary.to_csv(out, index=False)
print(f"Saved → {out}")

## 10. Interpretation guide

| Observation | Interpretation |
|---|---|
| Points of the **same CCD colour** deviate systematically in scienceFlux | Detector-level flat-field or gain offset |
| **All CCDs** deviate together at the same epoch | Time-variable zeropoint (atmosphere, throughput) |
| psfFlux (□) varies more than scienceFlux (●) | Template bias / template noise is the main source of variability |
| templateFlux (◇) drifts over time | Template epoch or depth is wrong; consider regenerating with Butler |
| psfFlux ≈ 0 but scienceFlux varies | Star truly variable — or photometric calibration issue |

**Next step:** cross-match with `visit_summary_src.csv` (seeing, sky background,
zeropoint) for the visits that show anomalous CCD-level behaviour.